# Robustness Analysis of LLM-Based Conversational Recommender Systems
## Applying RobustExplain Evaluation Metrics to ChatCRS

**Paper:** Preliminary Experiments for Robustness Analysis of ChatCRS  
**Dataset:** DuRecDial 2.0 (English subset)  
**Model:** LLaMA 3.1-8B via Groq API  
**Perturbations:** Noise Injection, Temporal Shuffle, Entity Corruption, Missing Values  

---

## Section 1: Setup & Installation

In [ ]:
# Install required libraries
!pip install groq pandas numpy scikit-learn -q
print("✅ Libraries installed!")

In [ ]:
# Import all libraries
import json
import random
import time
import os
import numpy as np
from groq import Groq
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

print("✅ All imports successful!")

## Section 2: API Configuration
> ⚠️ Replace `YOUR_GROQ_API_KEY` with your actual key from https://console.groq.com  
> Never share your API key publicly!

In [ ]:
# Configure Groq API
# Replace with your own key from console.groq.com
GROQ_API_KEY = "YOUR_GROQ_API_KEY"

client = Groq(api_key=GROQ_API_KEY)

# Quick connection test
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)
print("✅ API Connected:", response.choices[0].message.content)

## Section 3: Dataset Loading
We use the **DuRecDial 2.0** English subset — a human-annotated conversational recommendation dataset containing dialogue goals and knowledge triples.

In [ ]:
# Download DuRecDial 2.0 dataset
!git clone https://github.com/liuzeming01/DuRecDial.git -q
!unzip -q DuRecDial/data.zip -d DuRecDial/data_unzipped
print("✅ Dataset downloaded!")

In [ ]:
# Load English training dialogues
dialogues = []
with open('DuRecDial/data_unzipped/data/en_train.txt', 'r') as f:
    for line in f:
        dialogues.append(json.loads(line))

print(f"✅ Total dialogues loaded: {len(dialogues)}")
print(f"\nDataset fields: {list(dialogues[0].keys())}")
print(f"\nExample goal: {dialogues[0]['goal']}")
print(f"Example knowledge triple: {dialogues[0]['knowledge'][3]}")
print(f"Conversation turns: {len(dialogues[0]['conversation'])}")

In [ ]:
# Select 50 dialogues for experiment
sample = dialogues[:50]
print(f"✅ Working with {len(sample)} dialogues for robustness evaluation")

## Section 4: Perturbation Functions
We implement 4 perturbation types adapted from the **RobustExplain** framework (Zhang et al., 2026) for the CRS domain:

| Perturbation | Description | CRS Impact |
|---|---|---|
| **Noise Injection** | Adds random off-topic utterances | Confuses knowledge retrieval agent |
| **Temporal Shuffle** | Shuffles conversation turn order | Breaks goal planning sequence |
| **Entity Corruption** | Replaces entities with wrong ones | Retrieves completely wrong knowledge |
| **Missing Values** | Removes knowledge entries | Forces LLM to hallucinate |

In [ ]:
def inject_noise(dialogue):
    """Perturbation 1: Insert random off-topic utterances into conversation."""
    noisy = dialogue.copy()
    conversation = noisy['conversation'].copy()
    noise_sentences = [
        "By the way, what's the weather like today?",
        "I need to go buy groceries later.",
        "Did you watch the football game yesterday?",
        "I'm feeling a bit tired today.",
        "What time is it?"
    ]
    noise = random.choice(noise_sentences)
    position = random.randint(1, len(conversation) - 1)
    conversation.insert(position, noise)
    noisy['conversation'] = conversation
    return noisy


def temporal_shuffle(dialogue):
    """Perturbation 2: Shuffle conversation turn order (keep first and last)."""
    noisy = dialogue.copy()
    conversation = noisy['conversation'].copy()
    middle = conversation[1:-1]
    random.shuffle(middle)
    noisy['conversation'] = [conversation[0]] + middle + [conversation[-1]]
    return noisy


def entity_corruption(dialogue):
    """Perturbation 3: Replace real entities with incorrect but plausible ones."""
    noisy = dialogue.copy()
    conversation = noisy['conversation'].copy()
    replacements = {
        "Cecilia Cheung": "Jet Li",
        "Nicholas Tse": "Jackie Chan",
        "Failan": "The Matrix",
        "The Stool Pigeon": "Titanic",
        "Kris Wu": "Jay Chou"
    }
    new_conversation = []
    for turn in conversation:
        for original, replacement in replacements.items():
            turn = turn.replace(original, replacement)
        new_conversation.append(turn)
    noisy['conversation'] = new_conversation
    return noisy


def missing_values(dialogue):
    """Perturbation 4: Randomly remove 30% of knowledge entries."""
    noisy = dialogue.copy()
    knowledge = noisy['knowledge'].copy()
    new_knowledge = []
    for k in knowledge:
        if len(k) == 0 or random.random() > 0.3:
            new_knowledge.append(k)
        else:
            new_knowledge.append([])
    noisy['knowledge'] = new_knowledge
    return noisy


print("✅ All 4 perturbation functions defined!")

In [ ]:
# Apply all perturbations to 50 dialogues
perturbed_data = {
    'original':          sample,
    'noise_injection':   [inject_noise(d)      for d in sample],
    'temporal_shuffle':  [temporal_shuffle(d)  for d in sample],
    'entity_corruption': [entity_corruption(d) for d in sample],
    'missing_values':    [missing_values(d)    for d in sample],
}

for name, data in perturbed_data.items():
    print(f"✅ {name}: {len(data)} dialogues")

## Section 5: LLM Response Generation
We simulate ChatCRS's conversational agent by prompting LLaMA 3.1-8B with dialogue history and knowledge triples, following the same ICL prompt design described in Li et al. (2025).

In [ ]:
def get_llm_response(dialogue):
    """
    Simulate ChatCRS conversational agent.
    Sends dialogue history + knowledge triples to LLaMA
    and returns the generated recommendation response.
    """
    conversation_text = "\n".join(dialogue['conversation'])
    knowledge_text = ""
    for k in dialogue['knowledge']:
        if len(k) == 3:
            knowledge_text += f"- {k[0]} -> {k[1]} -> {k[2]}\n"

    prompt = f"""You are an excellent conversational recommender.

Dialogue History:
{conversation_text}

Relevant Knowledge:
{knowledge_text}

Based on the dialogue history and knowledge above, generate the next system response and recommend an item to the user."""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


print("✅ LLM response function defined!")

In [ ]:
# Run experiments across all 50 dialogues and all perturbation types
all_responses = {k: [] for k in perturbed_data.keys()}

print("Running experiments on 50 dialogues x 5 conditions = 250 API calls")
print("Estimated time: ~15-20 minutes\n")

for i in range(50):
    print(f"Processing dialogue {i+1}/50...", end='\r')
    for perturbation_name, data in perturbed_data.items():
        all_responses[perturbation_name].append(
            get_llm_response(data[i])
        )
    time.sleep(0.5)  # Respect API rate limits

print(f"\n✅ Experiments complete! Total responses: {50 * 5}")

## Section 6: Robustness Metrics
We implement the multi-dimensional robustness metrics from **RobustExplain** (Zhang et al., 2026):

- **Semantic Similarity (Sem):** Bag-of-words cosine similarity — measures meaning preservation  
- **Keyword Stability (Key):** Jaccard coefficient on key terms — measures entity/item consistency  
- **Length Stability (Len):** Relative length preservation — measures verbosity consistency  
- **Overall CRS-ρ:** Weighted composite score (Sem=0.4, Key=0.35, Len=0.25)

In [ ]:
def semantic_similarity(text1, text2):
    """Semantic Similarity (Sem): Bag-of-words cosine similarity."""
    vectorizer = CountVectorizer()
    try:
        vectors = vectorizer.fit_transform([text1, text2])
        return cosine_similarity(vectors[0], vectors[1])[0][0]
    except:
        return 0.0


def keyword_stability(text1, text2):
    """Keyword Stability (Key): Jaccard coefficient on content words."""
    stopwords = {'the','a','an','is','it','in','of','and','to','you',
                 'i','that','for','on','with','this','was','he','she',
                 'they','would','like','have','be','your'}
    words1 = set(text1.lower().split()) - stopwords
    words2 = set(text2.lower().split()) - stopwords
    if len(words1 | words2) == 0:
        return 0.0
    return len(words1 & words2) / len(words1 | words2)


def length_stability(text1, text2):
    """Length Stability (Len): Relative length preservation."""
    len1, len2 = len(text1), len(text2)
    return 1 - abs(len1 - len2) / max(len1, len2)


def compute_robustness(original_response, perturbed_response):
    """
    Combined CRS Robustness Score (CRS-rho).
    Weights: Sem=0.4, Key=0.35, Len=0.25 (from RobustExplain)
    """
    sem      = semantic_similarity(original_response, perturbed_response)
    key      = keyword_stability(original_response, perturbed_response)
    len_score = length_stability(original_response, perturbed_response)
    overall  = (0.4 * sem) + (0.35 * key) + (0.25 * len_score)
    return {
        'semantic': round(sem, 3),
        'keyword':  round(key, 3),
        'length':   round(len_score, 3),
        'overall':  round(overall, 3)
    }


print("✅ Robustness metrics defined!")

## Section 7: Results

In [ ]:
# Calculate average robustness scores across all 50 dialogues
final_results = {}

print("=" * 55)
print("  ROBUSTNESS EVALUATION RESULTS (50 dialogues)")
print("=" * 55)
print(f"{'Perturbation':<22} {'Sem':>6} {'Key':>6} {'Len':>6} {'Overall':>8}")
print("-" * 55)

for perturbation in ['noise_injection', 'temporal_shuffle',
                     'entity_corruption', 'missing_values']:
    sem_s, key_s, len_s, ovr_s = [], [], [], []
    for i in range(50):
        s = compute_robustness(
            all_responses['original'][i],
            all_responses[perturbation][i]
        )
        sem_s.append(s['semantic'])
        key_s.append(s['keyword'])
        len_s.append(s['length'])
        ovr_s.append(s['overall'])

    final_results[perturbation] = {
        'semantic': round(sum(sem_s)/50, 3),
        'keyword':  round(sum(key_s)/50, 3),
        'length':   round(sum(len_s)/50, 3),
        'overall':  round(sum(ovr_s)/50, 3)
    }
    r = final_results[perturbation]
    print(f"{perturbation:<22} {r['semantic']:>6} {r['keyword']:>6} {r['length']:>6} {r['overall']:>8}")

print("-" * 55)
avg = sum(v['overall'] for v in final_results.values()) / 4
print(f"{'Average':<22} {'':>6} {'':>6} {'':>6} {avg:>8.3f}")
print(f"{'RobustExplain baseline':<22} {'':>6} {'':>6} {'':>6} {'0.500':>8}")
print("=" * 55)

## Section 8: Save Results

In [ ]:
# Save all responses and results for later use
with open('all_responses.json', 'w') as f:
    json.dump(all_responses, f)

with open('final_results.json', 'w') as f:
    json.dump(final_results, f)

print("✅ Results saved to all_responses.json and final_results.json")
print(f"✅ Total responses saved: {sum(len(v) for v in all_responses.values())}")

In [ ]:
# Download files to your computer
from google.colab import files
files.download('all_responses.json')
files.download('final_results.json')
print("✅ Files downloaded!")

In [ ]:
# Optional: Also backup to Google Drive
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copy('all_responses.json', '/content/drive/MyDrive/all_responses.json')
shutil.copy('final_results.json', '/content/drive/MyDrive/final_results.json')
print("✅ Backed up to Google Drive!")

## Section 9: Reload Saved Results (Run this next time instead of re-running experiments)

In [ ]:
# Upload saved files and reload
from google.colab import files
uploaded = files.upload()  # Select all_responses.json and final_results.json

with open('all_responses.json', 'r') as f:
    all_responses = json.load(f)
with open('final_results.json', 'r') as f:
    final_results = json.load(f)

print(f"✅ Reloaded {len(all_responses['original'])} dialogues")
print(f"✅ Results: {list(final_results.keys())}")